# 08 组合构建：把分数变成持仓

## 8.1 本章要解决什么问题

前两章已经把缓存 ETF 行情变成了因子值、综合分数和简单的有效性检查。本章开始回答组合层的问题：

- 在哪些日期更新持仓：用交易日历找出月末调仓日。
- 买哪些标的：按综合分数做 TopN 选择。
- 每个标的配多少：从最透明的等权开始。
- 怎么给回测使用：把稀疏调仓权重展开成完整每日目标权重矩阵。
- 怎么检查组合动作：观察现金/未投资权重、换手率和权重完整性。

## 8.2 工作流位置

`缓存 ETF 行情 -> 因子分数 -> 组合目标权重 -> 回测 -> 绩效报告 -> 交易信号`

本章的输入是缓存 ETF 数据与综合分数，输出是目标权重。下一章 `09_backtest_engine_manual_and_bt.ipynb` 会把这些权重放进回测口径里，处理滞后持仓、收益、成本和 `bt` 框架复现。

## 8.3 本章产物

为了不覆盖后续流水线的通用产物，本章只写入带章节名前缀的文件：

- `outputs/results/chapter08_sparse_rebalance_weights.csv`
- `outputs/results/chapter08_target_weight_matrix.csv`
- `outputs/results/chapter08_portfolio_state.csv`
- `outputs/results/chapter08_latest_target_weights.csv`


In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    candidates = [start.resolve(), *start.resolve().parents]
    candidates += [candidate / "pyquant-roadmap" for candidate in candidates]
    for candidate in candidates:
        if (candidate / "lib").exists() and (candidate / "notebooks").exists():
            return candidate.resolve()
    raise RuntimeError("Cannot find pyquant-roadmap project root from the current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from lib.paths import RESULTS_DIR
from lib.data import load_sample_assets, load_sample_prices
from lib.factors import build_technical_factor_panel, combine_score, zscore_by_date

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

TOP_N = 3
FACTOR_WEIGHTS = {"momentum_60": 0.45, "low_vol_20": 0.35, "ma_gap_20_60": 0.20}
FACTOR_COLS = list(FACTOR_WEIGHTS)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

prices = load_sample_prices().copy()
prices["date"] = pd.to_datetime(prices["date"])
prices["code"] = prices["code"].astype(str)

assets = load_sample_assets().copy()
assets["code"] = assets["code"].astype(str)

asset_order = [code for code in assets["code"] if code in set(prices["code"])]
close = (
    prices.pivot_table(index="date", columns="code", values="close", aggfunc="last")
    .sort_index()
    .reindex(columns=asset_order)
)
close.index.name = "date"
close.columns.name = "code"

setup_summary = pd.Series(
    {
        "project_root": ".",
        "price_rows": len(prices),
        "assets": len(asset_order),
        "start_date": close.index.min().date(),
        "end_date": close.index.max().date(),
        "top_n": TOP_N,
    },
    name="value",
)
display(setup_summary.to_frame())
display(assets[["code", "name", "asset_type"]])


,value
project_root,.
price_rows,2900
assets,4
start_date,2021-01-04
end_date,2023-12-29
top_n,3


,code,name,asset_type
0,510300,沪深300ETF,ETF
1,510500,中证500ETF,ETF
2,159915,创业板ETF,ETF
3,512100,中证1000ETF,ETF


## 8.4 先用小例子看清核心动作

组合构建不用一开始就上优化器。先把问题拆成四步：

1. 每个交易日都有一张分数表：`date, code, score`。
2. 只在调仓日读取分数，非调仓日沿用旧权重。
3. 调仓日按分数排序，取 TopN。
4. 入选标的等权，未入选标的权重为 0。

下面的小表是手写数据，目的是让每一行都能被肉眼检查。


In [2]:
tiny_scores = pd.DataFrame(
    {
        "date": pd.to_datetime(
            [
                "2024-01-29", "2024-01-29", "2024-01-29",
                "2024-01-31", "2024-01-31", "2024-01-31",
                "2024-02-28", "2024-02-28", "2024-02-28",
            ]
        ),
        "code": ["A", "B", "C", "A", "B", "C", "A", "B", "C"],
        "score": [0.20, 0.80, 0.40, 0.10, 0.90, 0.30, 0.95, 0.15, 0.55],
    }
)

display(tiny_scores)


,date,code,score
0,2024-01-29,A,0.20
1,2024-01-29,B,0.80
2,2024-01-29,C,0.40
3,2024-01-31,A,0.10
4,2024-01-31,B,0.90
5,2024-01-31,C,0.30
6,2024-02-28,A,0.95
7,2024-02-28,B,0.15
8,2024-02-28,C,0.55


## 8.5 调仓日：用交易日定义月末

低频策略常用“月末调仓”，但月末不一定是自然月最后一天，而是当月最后一个有分数、可交易的交易日。这里先写一个最小函数：把所有日期按月份分组，取每个月最大交易日。


In [3]:
def month_end_trading_dates(dates) -> pd.DatetimeIndex:
    unique_dates = pd.Series(pd.to_datetime(pd.Index(dates).unique())).sort_values()
    month_end = unique_dates.groupby(unique_dates.dt.to_period("M")).max()
    return pd.DatetimeIndex(month_end.to_numpy(), name="date")


tiny_rebalance_dates = month_end_trading_dates(tiny_scores["date"])
print("tiny rebalance dates:", [d.date().isoformat() for d in tiny_rebalance_dates])


tiny rebalance dates: ['2024-01-31', '2024-02-28']


## 8.6 TopN 选择和等权

TopN 只回答“谁进入组合”。等权再回答“每个入选标的分多少预算”。

如果某个调仓日只有 2 个合格标的，而参数是 Top3，本章的默认做法是把这 2 个标的重新等权到 100%。在更真实的执行层，如果有单票上限、整数份取整或不可成交，剩余预算会体现为现金权重。


In [4]:
def top_n_equal_weight_manual(scores: pd.DataFrame, n: int = 3, score_col: str = "score") -> pd.DataFrame:
    if n <= 0:
        raise ValueError("n must be positive")

    work = scores.dropna(subset=[score_col]).copy()
    work["date"] = pd.to_datetime(work["date"])
    work["code"] = work["code"].astype(str)

    rows = []
    for dt, group in work.groupby("date", sort=True):
        top = group.sort_values([score_col, "code"], ascending=[False, True]).head(n).copy()
        if top.empty:
            continue
        top["rank"] = range(1, len(top) + 1)
        top["weight"] = 1.0 / len(top)
        rows.append(top[["date", "code", score_col, "rank", "weight"]])

    if not rows:
        return pd.DataFrame(columns=["date", "code", score_col, "rank", "weight"])
    return pd.concat(rows, ignore_index=True).sort_values(["date", "rank"]).reset_index(drop=True)


tiny_sparse_weights = top_n_equal_weight_manual(
    tiny_scores[tiny_scores["date"].isin(tiny_rebalance_dates)],
    n=2,
)
display(tiny_sparse_weights)


,date,code,score,rank,weight
0,2024-01-31,B,0.90,1,0.5
1,2024-01-31,C,0.30,2,0.5
2,2024-02-28,A,0.95,1,0.5
3,2024-02-28,C,0.55,2,0.5


## 8.7 稀疏权重 vs 完整每日权重矩阵

`tiny_sparse_weights` 是稀疏表：只有调仓日、入选标的和权重。它适合审查“每次调仓选了谁”。

回测更常需要完整矩阵：每个交易日一行、每个资产一列。非调仓日如果不做 carry-forward，就会是 0；如果做 carry-forward，就表示继续持有上一次调仓后的目标权重。


In [5]:
def weights_to_matrix_manual(
    weights: pd.DataFrame,
    index: pd.Index,
    columns: pd.Index,
    carry_forward: bool = False,
) -> pd.DataFrame:
    index = pd.DatetimeIndex(index, name="date")
    columns = pd.Index(columns, name="code")

    if weights.empty:
        return pd.DataFrame(0.0, index=index, columns=columns)

    sparse = weights.pivot_table(index="date", columns="code", values="weight", aggfunc="sum")
    matrix = sparse.reindex(columns=columns).reindex(index=index)
    if carry_forward:
        matrix = matrix.ffill()
    return matrix.fillna(0.0)


tiny_calendar = pd.DatetimeIndex(pd.to_datetime(["2024-01-29", "2024-01-30", "2024-01-31", "2024-02-01", "2024-02-28"]))
tiny_columns = pd.Index(["A", "B", "C"], name="code")

tiny_rebalance_only = weights_to_matrix_manual(tiny_sparse_weights, tiny_calendar, tiny_columns, carry_forward=False)
tiny_carried = weights_to_matrix_manual(tiny_sparse_weights, tiny_calendar, tiny_columns, carry_forward=True)

display(pd.concat({"rebalance_only": tiny_rebalance_only, "carry_forward": tiny_carried}, axis=1).round(3))


rebalance_only           carry_forward          
code                    A    B    C             A    B    C
date                                                       
2024-01-29            0.0  0.0  0.0           0.0  0.0  0.0
2024-01-30            0.0  0.0  0.0           0.0  0.0  0.0
2024-01-31            0.0  0.5  0.5           0.0  0.5  0.5
2024-02-01            0.0  0.0  0.0           0.0  0.5  0.5
2024-02-28            0.5  0.0  0.5           0.5  0.5  0.5

## 8.8 现金/未投资权重和换手

每日目标权重矩阵还有两个必须检查的派生量：

- `invested_weight`：所有资产权重之和。
- `cash_weight`：`1 - invested_weight`。如果第一天还没到调仓日，它应该是 1；如果有单票上限、取整或不可成交，也可能大于 0。
- `gross_turnover`：本项目采用 `sum(abs(delta_weight))` 作为“成交权重”的近似。完整从 A 切到 B 会记作 2.0，因为买卖两边都可能产生费用。下一章的成本模型会沿用这个口径。


In [6]:
def turnover_manual(weight_matrix: pd.DataFrame) -> pd.Series:
    turnover = weight_matrix.diff().abs().sum(axis=1)
    if not turnover.empty:
        turnover.iloc[0] = weight_matrix.iloc[0].abs().sum()
    return turnover.fillna(0.0).rename("gross_turnover")


def portfolio_state_from_weights(weight_matrix: pd.DataFrame) -> pd.DataFrame:
    invested = weight_matrix.sum(axis=1).rename("invested_weight")
    state = pd.DataFrame(
        {
            "invested_weight": invested,
            "cash_weight": (1.0 - invested).clip(lower=0.0),
            "gross_turnover": turnover_manual(weight_matrix),
        }
    )
    return state


display(portfolio_state_from_weights(tiny_carried).round(3))


,invested_weight,cash_weight,gross_turnover
date,,,
2024-01-29,0.0,1.0,0.0
2024-01-30,0.0,1.0,0.0
2024-01-31,1.0,0.0,1.0
2024-02-01,1.0,0.0,0.0
2024-02-28,1.5,0.0,0.5


## 8.9 从缓存 ETF 数据得到实际分数

现在回到项目主线。样本 ETF 行情来自 `data/sample/prices.parquet`，是前面章节通过 AKShare 路径缓存下来的本地数据。

本章不要求第 07 章已经写出通用分数文件。为了支持重复运行，如果 `outputs/results/factor_scores.csv` 已经由第 10-11 章的流水线生成，本章会直接读取；如果文件不存在，就用同一份缓存 ETF 行情重新计算技术因子、横截面标准化和综合分数。这样既能保持章节独立，也能让后续工程化流水线复用同一套口径。


In [7]:
def build_scores_from_cached_prices(price_panel: pd.DataFrame) -> pd.DataFrame:
    factor_panel = build_technical_factor_panel(price_panel)
    scored_panel = combine_score(zscore_by_date(factor_panel, FACTOR_COLS), FACTOR_WEIGHTS)
    return scored_panel.sort_values(["date", "code"]).reset_index(drop=True)


factor_scores_path = RESULTS_DIR / "factor_scores.csv"
required_score_cols = {"date", "code", "score"}

if factor_scores_path.exists():
    loaded_scores = pd.read_csv(factor_scores_path, parse_dates=["date"], dtype={"code": str})
    if required_score_cols.issubset(loaded_scores.columns):
        scored = loaded_scores.copy()
        score_source = factor_scores_path.relative_to(PROJECT_ROOT).as_posix()
    else:
        scored = build_scores_from_cached_prices(prices)
        score_source = "rebuilt from data/sample/prices.parquet because factor_scores.csv was incomplete"
else:
    scored = build_scores_from_cached_prices(prices)
    score_source = "rebuilt from data/sample/prices.parquet"

scored["date"] = pd.to_datetime(scored["date"])
scored["code"] = scored["code"].astype(str)
scored = scored.sort_values(["date", "code"]).reset_index(drop=True)

score_summary = pd.Series(
    {
        "score_source": score_source,
        "score_rows": len(scored),
        "first_score_date": scored["date"].min().date(),
        "last_score_date": scored["date"].max().date(),
        "score_columns": ", ".join([col for col in [*FACTOR_COLS, "score"] if col in scored.columns]),
    },
    name="value",
)

display(score_summary.to_frame())
display(scored.head().round(4))


,value
score_source,outputs/results/factor_scores.csv
score_rows,2656
first_score_date,2021-04-08
last_score_date,2023-12-29
score_columns,"momentum_60, low_vol_20, ma_gap_20_60, score"


,date,code,momentum_60,low_vol_20,ma_gap_20_60,momentum_60_z,low_vol_20_z,ma_gap_20_60_z,score
0,2021-04-08,159915,-0.0904,-0.0176,-0.0881,-1.5948,-1.4483,-1.5026,-1.5251
1,2021-04-08,510300,-0.0339,-0.0143,-0.0556,0.5301,-0.3345,-0.3013,0.0612
2,2021-04-08,510500,-0.0191,-0.0096,-0.0259,1.0842,1.2087,0.7919,1.0693
3,2021-04-08,512100,-0.0485,-0.0115,-0.0199,-0.0195,0.5740,1.0120,0.3946
4,2021-04-09,159915,-0.0870,-0.0174,-0.0843,-1.5006,-1.4121,-1.4783,-1.4652


## 8.10 实盘化第一步：确定月末调仓日期

这里用“分数表里的最后一个交易日”定义每个月的调仓日。这样可以避免自然月末遇到周末、节假日或没有足够历史窗口的问题。


In [8]:
rebalance_dates = month_end_trading_dates(scored["date"])
rebalance_dates = pd.DatetimeIndex([dt for dt in rebalance_dates if dt in close.index], name="date")

rebalance_preview = pd.DataFrame(
    {
        "first_5_rebalance_dates": [d.date().isoformat() for d in rebalance_dates[:5]],
        "last_5_rebalance_dates": [d.date().isoformat() for d in rebalance_dates[-5:]],
    }
)

print(f"rebalance count: {len(rebalance_dates)}")
display(rebalance_preview)


rebalance count: 33


,first_5_rebalance_dates,last_5_rebalance_dates
0,2021-04-30,2023-08-31
1,2021-05-31,2023-09-28
2,2021-06-30,2023-10-31
3,2021-07-30,2023-11-30
4,2021-08-31,2023-12-29


## 8.11 用 `lib.portfolio` 生成稀疏调仓权重

手写函数用于学习，项目函数用于复用。`lib.portfolio.top_n_equal_weight` 的输入是一张长表：`date, code, score`；输出是稀疏目标权重：只保留调仓日入选标的。

这一步仍然只是“目标权重”，还不是实际成交持仓。整数份、涨跌停、停牌和成本会在后面的回测与交易信号章节继续处理。


In [9]:
from lib.portfolio import calculate_turnover, top_n_equal_weight, weights_to_matrix

rebalance_scores = scored[scored["date"].isin(rebalance_dates)].copy()
rebalance_scores["score_rank"] = rebalance_scores.groupby("date")["score"].rank(ascending=False, method="first")

sparse_weights = top_n_equal_weight(rebalance_scores, n=TOP_N)
sparse_weights["date"] = pd.to_datetime(sparse_weights["date"])
sparse_weights["code"] = sparse_weights["code"].astype(str)

sparse_with_details = (
    sparse_weights.merge(
        rebalance_scores[["date", "code", "score", "score_rank", *[c for c in FACTOR_COLS if c in rebalance_scores.columns]]],
        on=["date", "code"],
        how="left",
    )
    .merge(assets[["code", "name"]], on="code", how="left")
    .sort_values(["date", "score_rank", "code"])
    .reset_index(drop=True)
)

print(f"sparse rows: {len(sparse_with_details)}")
display(sparse_with_details.tail(12).round(4))


sparse rows: 99


,date,code,weight,score,score_rank,momentum_60,low_vol_20,ma_gap_20_60,name
87,2023-09-28,510300,0.3333,1.0029,1.0,-0.0356,-0.0088,-0.0208,沪深300ETF
88,2023-09-28,510500,0.3333,0.8227,2.0,-0.0551,-0.0083,-0.0245,中证500ETF
89,2023-09-28,512100,0.3333,-0.5644,3.0,-0.0834,-0.0096,-0.0294,中证1000ETF
90,2023-10-31,512100,0.3333,0.7214,1.0,-0.0749,-0.0115,-0.0276,中证1000ETF
91,2023-10-31,510500,0.3333,0.4795,2.0,-0.0874,-0.0105,-0.0330,中证500ETF
92,2023-10-31,510300,0.3333,0.0111,3.0,-0.1098,-0.0090,-0.0392,沪深300ETF
93,2023-11-30,512100,0.3333,0.8328,1.0,0.0029,-0.0096,0.0125,中证1000ETF
94,2023-11-30,510500,0.3333,0.4040,2.0,-0.0387,-0.0077,-0.0046,中证500ETF
95,2023-11-30,510300,0.3333,-0.2121,3.0,-0.0841,-0.0068,-0.0204,沪深300ETF
96,2023-12-29,512100,0.3333,0.8990,1.0,-0.0387,-0.0104,-0.0130,中证1000ETF


## 8.12 展开成完整每日目标权重矩阵

稀疏权重适合审查，完整矩阵适合回测。下面同时展示两种矩阵：

- `rebalance_only_matrix`：只有调仓日有权重，非调仓日为 0。
- `target_weight_matrix`：调仓日之后向前填充，表示继续持有上一期目标组合。

注意：这里的 carry-forward 是目标权重路径，不是价格漂移后的实际权重。真实回测会在下一章用滞后持仓和收益计算进一步处理。


In [10]:
rebalance_only_matrix = weights_to_matrix(sparse_weights, close.index, close.columns, carry_forward=False)
target_weight_matrix = weights_to_matrix(sparse_weights, close.index, close.columns, carry_forward=True)

target_weight_matrix.index.name = "date"
target_weight_matrix.columns.name = "code"
rebalance_only_matrix.index.name = "date"
rebalance_only_matrix.columns.name = "code"

first_rebalance = sparse_weights["date"].min()
window = close.index[(close.index >= first_rebalance - pd.Timedelta(days=3)) & (close.index <= first_rebalance + pd.Timedelta(days=8))]

comparison_window = pd.concat(
    {"rebalance_only": rebalance_only_matrix.loc[window], "carry_forward": target_weight_matrix.loc[window]},
    axis=1,
)

display(comparison_window.round(4))


rebalance_only                        carry_forward                       
code               510300  510500 159915  512100        510300  510500 159915  512100
date                                                                                 
2021-04-27         0.0000  0.0000    0.0  0.0000        0.0000  0.0000    0.0  0.0000
2021-04-28         0.0000  0.0000    0.0  0.0000        0.0000  0.0000    0.0  0.0000
2021-04-29         0.0000  0.0000    0.0  0.0000        0.0000  0.0000    0.0  0.0000
2021-04-30         0.3333  0.3333    0.0  0.3333        0.3333  0.3333    0.0  0.3333
2021-05-06         0.0000  0.0000    0.0  0.0000        0.3333  0.3333    0.0  0.3333
2021-05-07         0.0000  0.0000    0.0  0.0000        0.3333  0.3333    0.0  0.3333

## 8.13 检查现金、权重和换手

组合权重不是生成完就结束。至少要检查三件事：

- 权重和是否超过 1。
- 第一笔调仓前是否保持现金。
- 换手是否只集中在调仓日附近。

如果这些检查过不了，后面的回测结果再漂亮也不可信。


In [11]:
portfolio_state = portfolio_state_from_weights(target_weight_matrix)
portfolio_state["gross_turnover"] = calculate_turnover(target_weight_matrix)
portfolio_state.index.name = "date"

nonzero_turnover = portfolio_state[portfolio_state["gross_turnover"] > 0]
quality_summary = pd.Series(
    {
        "max_invested_weight": portfolio_state["invested_weight"].max(),
        "min_cash_weight": portfolio_state["cash_weight"].min(),
        "max_gross_turnover": portfolio_state["gross_turnover"].max(),
        "turnover_days": len(nonzero_turnover),
        "avg_turnover_on_turnover_days": nonzero_turnover["gross_turnover"].mean(),
    },
    name="value",
)

display(portfolio_state.loc[window].round(4))
display(quality_summary.to_frame().round(4))


,invested_weight,cash_weight,gross_turnover
date,,,
2021-04-27,0.0,1.0,0.0
2021-04-28,0.0,1.0,0.0
2021-04-29,0.0,1.0,0.0
2021-04-30,1.0,0.0,1.0
2021-05-06,1.0,0.0,0.0
2021-05-07,1.0,0.0,0.0


,value
max_invested_weight,1.0000
min_cash_weight,0.0000
max_gross_turnover,1.0000
turnover_days,14.0000
avg_turnover_on_turnover_days,0.6905


## 8.14 组合质量断言

把关键约束写成断言，比只看表格更可靠。本章先检查最基础的目标权重约束；真实交易约束会在后续章节逐步加入。


In [12]:
selected_count = sparse_weights.groupby("date")["code"].nunique().rename("selected_count")
weight_sum_by_rebalance = sparse_weights.groupby("date")["weight"].sum().rename("weight_sum")
unknown_codes = sorted(set(sparse_weights["code"]) - set(close.columns))

assert not sparse_weights.empty, "No sparse weights were generated."
assert not unknown_codes, f"Weights contain codes not found in the close matrix: {unknown_codes}"
assert portfolio_state["invested_weight"].max() <= 1.0 + 1e-9, "Target weights exceed 100%."
assert (weight_sum_by_rebalance <= 1.0 + 1e-9).all(), "Sparse rebalance weights exceed 100%."
assert selected_count.max() <= TOP_N, "A rebalance date selected more than TOP_N assets."

rebalance_checks = pd.concat([selected_count, weight_sum_by_rebalance], axis=1).tail(10)
display(rebalance_checks.round(4))


,selected_count,weight_sum
date,,
2023-03-31,3,1.0
2023-04-28,3,1.0
2023-05-31,3,1.0
2023-06-30,3,1.0
2023-07-31,3,1.0
2023-08-31,3,1.0
2023-09-28,3,1.0
2023-10-31,3,1.0
2023-11-30,3,1.0


## 8.15 保存本章有用输出

这里保存的是“组合构建教学产物”，文件名都带 `chapter08_` 前缀。后面的工程化流水线会继续生成通用的 `target_weights.csv`、订单建议和报告文件。


In [13]:
latest_matrix_date = target_weight_matrix.index.max()
latest_rebalance_date = sparse_weights["date"].max()

latest_target_weights = (
    target_weight_matrix.loc[latest_matrix_date]
    .rename("target_weight")
    .reset_index()
    .query("target_weight > 0")
    .merge(assets[["code", "name"]], on="code", how="left")
    .merge(
        rebalance_scores.loc[rebalance_scores["date"].eq(latest_rebalance_date), ["code", "score", "score_rank"]],
        on="code",
        how="left",
    )
    .sort_values("target_weight", ascending=False)
    .reset_index(drop=True)
)
latest_target_weights.insert(0, "date", latest_matrix_date)
latest_target_weights.insert(1, "rebalance_source_date", latest_rebalance_date)

paths = {
    "sparse_rebalance_weights": RESULTS_DIR / "chapter08_sparse_rebalance_weights.csv",
    "target_weight_matrix": RESULTS_DIR / "chapter08_target_weight_matrix.csv",
    "portfolio_state": RESULTS_DIR / "chapter08_portfolio_state.csv",
    "latest_target_weights": RESULTS_DIR / "chapter08_latest_target_weights.csv",
}

sparse_with_details.to_csv(paths["sparse_rebalance_weights"], index=False, encoding="utf-8-sig")
target_weight_matrix.to_csv(paths["target_weight_matrix"], encoding="utf-8-sig")
portfolio_state.to_csv(paths["portfolio_state"], encoding="utf-8-sig")
latest_target_weights.to_csv(paths["latest_target_weights"], index=False, encoding="utf-8-sig")

saved = pd.DataFrame(
    [
        {"artifact": name, "path": path.relative_to(PROJECT_ROOT).as_posix(), "rows": len(pd.read_csv(path))}
        for name, path in paths.items()
    ]
)

display(latest_target_weights.round(4))
display(saved)


,date,rebalance_source_date,code,target_weight,name,score,score_rank
0,2023-12-29,2023-12-29,510300,0.3333,沪深300ETF,-0.7652,3.0
1,2023-12-29,2023-12-29,510500,0.3333,中证500ETF,0.6667,2.0
2,2023-12-29,2023-12-29,512100,0.3333,中证1000ETF,0.8990,1.0


,artifact,path,rows
0,sparse_rebalance_weights,outputs/results/chapter08_sparse_rebalance_wei...,99
1,target_weight_matrix,outputs/results/chapter08_target_weight_matrix...,725
2,portfolio_state,outputs/results/chapter08_portfolio_state.csv,725
3,latest_target_weights,outputs/results/chapter08_latest_target_weight...,3


## 8.16 练习：把 TopN 从 3 改成 2

练习目标：观察更集中持仓会如何改变换手和持仓数量。

下面给出答案脚手架。你可以把 `EXERCISE_TOP_N` 改成 1、2、4，再比较 `avg_turnover_on_turnover_days` 和 `selected_count`。


In [14]:
EXERCISE_TOP_N = 2

exercise_sparse = top_n_equal_weight(rebalance_scores, n=EXERCISE_TOP_N)
exercise_matrix = weights_to_matrix(exercise_sparse, close.index, close.columns, carry_forward=True)
exercise_state = portfolio_state_from_weights(exercise_matrix)
exercise_state["gross_turnover"] = calculate_turnover(exercise_matrix)
exercise_nonzero_turnover = exercise_state[exercise_state["gross_turnover"] > 0]
exercise_selected_count = exercise_sparse.groupby("date")["code"].nunique()

exercise_summary = pd.Series(
    {
        "exercise_top_n": EXERCISE_TOP_N,
        "avg_selected_count": exercise_selected_count.mean(),
        "max_invested_weight": exercise_state["invested_weight"].max(),
        "avg_turnover_on_turnover_days": exercise_nonzero_turnover["gross_turnover"].mean(),
        "latest_weight_sum": exercise_matrix.loc[exercise_matrix.index.max()].sum(),
    },
    name="value",
)

display(exercise_summary.to_frame().round(4))


,value
exercise_top_n,2.0
avg_selected_count,2.0
max_invested_weight,1.0
avg_turnover_on_turnover_days,1.0
latest_weight_sum,1.0


## 8.17 常见坑与下一章交接

常见坑：

- 把自然月末当成交易日。正确做法是从有数据的交易日里取每个月最后一天。
- 只保存稀疏调仓表，却忘了回测需要完整每日权重矩阵。
- 非调仓日没有 carry-forward，导致组合每天自动清仓。
- 把目标权重当成实际成交持仓。目标权重只表达“想买什么”，成交、成本、现金和不可交易约束还要由回测或交易模块处理。
- 不说明换手口径。本文和项目函数使用 `sum(abs(delta_weight))`，它更贴近下一章按成交权重扣成本的简化模型。

交接给第 09 章：本章得到的 `target_weight_matrix` 是“目标权重路径”。第 09 章会把它滞后一日用于收益计算，并比较手写最小回测与 `bt` 权重型回测。
